# A bounded application to published LeakageBench aggregates

This notebook executes an **analysis of actual published aggregate values**, not a new model evaluation. The numerical inputs come from Table 3 of LeakageBench, arXiv:2609.02207v1, retrieved on 23 September 2026. A hypothetical update withholds page outputs failing the paper's reported validity predicate, leaving every accepted prediction unchanged. Exact integer feasibility determines what those aggregates support about critical DocLeak among the remaining applicable pages.

The analysis never invents a per-page prediction file or treats compatible count completions as observations. It uses all 14 Table 3 rows, fixes the proposed validity gate before enumerating outcomes, and includes all five systems with reported validity below one. The result need not reverse a ranking or worsen a failure rate. Under this particular empty-output scoring convention, removing invalid outputs can only remove matching failures from applicable pages.

**Scope:** These are finite challenge-set bounds for the paper's typed-IoU matching loss. They are neither confidence intervals, estimated deployment probabilities, final-artifact concealment rates, nor privacy certificates. The challenge set has 500 pages from 291 source documents; pages are not asserted to be independent person-level or document-level release units.

The notebook uses only the Python standard library. Its embedded minimal factual input permits offline execution. It writes copies of the input/provenance to `public_protocol_inputs/` and results to `public_protocol_outputs/`. The full primary paper is not redistributed.

In [1]:
from pathlib import Path
from fractions import Fraction
from functools import lru_cache
from html.parser import HTMLParser
import csv, hashlib, io, json, math, platform, urllib.request
INPUT = Path("public_protocol_inputs")
OUT = Path("public_protocol_outputs")
INPUT.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
MINIMAL_CSV = 'system,ocr,docleak_crit,valid\r\nPresidio,Tesseract,0.990,0.996\r\nPresidio-best (GLiNER restr.),Textract,0.986,1.000\r\nGLiNER-base,Textract,0.978,1.000\r\nGLiNER-multi-PII,Textract,0.972,1.000\r\nNVIDIA GLiNER PII,Textract,0.986,1.000\r\nGLiNER2,Textract,0.994,0.998\r\nAmazon Comprehend PII,Textract,0.968,1.000\r\nOpenAI Privacy Filter,Textract,0.990,0.806\r\nGoogle DLP,Vendor,1.000,1.000\r\nQwen3-VL-32B,—,0.998,0.666\r\nInternVL3-38B,—,1.000,0.540\r\nGPT-5.4,—,1.000,1.000\r\nGPT-5.5,—,0.990,1.000\r\nGPT-5.5 + Code Interpreter,—,0.968,1.000\r\n'
PROVENANCE = {'source_title': 'LeakageBench: Document-Level Leakage Risk for Redacting Personally Identifiable Information in Document Images', 'source_authors': ['Vishnu Prasad Vijaya Kumar', 'Santhosh Venkatesh', 'Ivan P. Yamshchikov'], 'source_id': 'arXiv:2609.02207v1', 'source_version_date': '2026-09-02', 'retrieved_date_utc': '2026-09-23', 'source_url': 'https://arxiv.org/html/2609.02207v1', 'source_table_id': 'S6.T3', 'source_table_label': 'Table 3, full-schema results at IoU 0.75', 'source_html_sha256': '0664b482d98c482ddb81f92264a8a0514917ec2ee4f25da74ea85e5681eb2743', 'source_html_bytes': 458024, 'minimal_input_sha256': '19a6371e952b5d0b50d9305e57acba03055ab167eae644583273db1bac1f9076', 'total_pages': 500, 'source_documents': 291, 'source_rounding_convention_explicitly_documented': False, 'primary_analysis_display_assumption': 'Nearest 0.001, inclusive half-unit endpoints; both tie conventions admitted', 'sensitivity_display_assumption': 'Truncation to 0.001', 'applicability_assumption': 'Common full-schema Direct+Linkage applicable-page set for all Table 3 systems', 'validity_assumption': 'The reported page-validity indicator is the page-level unusable-output predicate; its false outcomes are scored as empty prediction sets', 'source_limitations': 'Challenge-set page counts and typed IoU matching; no deployment sampling claim, no final-artifact containment measurement', 'redistribution': 'Only numerical facts, system labels, original code and analysis are included. Full paper HTML is not redistributed.', 'primary_source_locators': [{'locator': 'Section 3.3', 'paraphrase': 'Headline DocLeak uses same-type matching at an IoU threshold; it is not complete-mask containment.'}, {'locator': 'Section 3.4, equation (1)', 'paraphrase': 'Each DocLeak denominator consists of pages containing an in-scope gold identifier.'}, {'locator': 'Section 6, introduction and Table 3 caption', 'paraphrase': 'Full-schema evaluation retains unsupported labels as misses. Page validity describes parseable, schema-conformant output; malformed or unusable outputs are scored as empty predictions.'}, {'locator': 'Table 4 and its caption', 'paraphrase': 'The separate supported-type evaluation changes the gold scope by system and is excluded from this reconstruction.'}, {'locator': 'Section 4.4, Table 2', 'paraphrase': 'The corpus has 500 pages from 291 source documents.'}, {'locator': 'Appendix B, OCR-free conversion and validation', 'paraphrase': 'The invalid-output conversion is described before scoring, without model-specific repair.'}], 'short_primary_quotes': ['Valid denotes parseable, schema-conformant page outputs.', 'Unsupported types and invalid outputs count as misses', 'pages with at least one in-scope identifier']}
assert hashlib.sha256(MINIMAL_CSV.encode("utf-8")).hexdigest() == PROVENANCE["minimal_input_sha256"]
(INPUT/"reported_metrics.csv").write_bytes(MINIMAL_CSV.encode("utf-8"))
(INPUT/"source_provenance.json").write_text(json.dumps(PROVENANCE,indent=2)+"\n")
REPORTED = list(csv.DictReader(io.StringIO(MINIMAL_CSV)))
assert len(REPORTED) == 14 and len({row["system"] for row in REPORTED}) == 14
N = PROVENANCE["total_pages"]
print(f"Loaded {len(REPORTED)} actual published system rows; source {PROVENANCE['source_id']}.")
print("Input SHA256:", PROVENANCE["minimal_input_sha256"])


Loaded 14 actual published system rows; source arXiv:2609.02207v1.
Input SHA256: 19a6371e952b5d0b50d9305e57acba03055ab167eae644583273db1bac1f9076


## Source verification and assumptions

The downloaded versioned HTML was parsed to obtain the input values, rather than manually estimating numbers from a plot. Table 3 is explicitly a full-schema comparison. Section 3.4 defines critical applicability through the gold Direct/Linkage labels, so the same applicable-page set is used across its systems. Table 4's model-dependent supported-type denominators are excluded. Section 6 and Appendix B describe the page-level unusable-output conversion to empty predictions, and Table 3 reports the corresponding validity indicator. Individual box rejection is distinct from this page gate.

The reconstruction assumes this reported page-validity predicate is the one triggering the empty-output treatment. If an implementation instead retained partial predictions on a page counted as invalid, the implication used below would need to be checked from logs and these bounds would not automatically apply. No such implementation logs are assumed available here.

The source displays three decimal places but does not specify its formatting routine. The primary analysis admits rates within half a displayed unit, including both tie conventions. A truncation sensitivity is supplied separately. A second sensitivity uses only the selected model's row, dropping all cross-model information about the common denominator.

The following cell checks the original HTML if the local, non-distributed cache exists. Remote verification is optional and disabled by default. A changed HTML hash can reflect a renderer change; numeric table equality is checked separately, and the original snapshot hash remains recorded.

In [2]:
class PrimaryTableParser(HTMLParser):
    def __init__(self):
        super().__init__(); self.active=False; self.cell=None; self.row=[]; self.rows=[]
    def handle_starttag(self,tag,attrs):
        if tag == "figure" and dict(attrs).get("id") == "S6.T3": self.active=True
        if self.active and tag == "tr": self.row=[]
        if self.active and tag in ("td","th"): self.cell=[]
    def handle_data(self,data):
        if self.active and self.cell is not None: self.cell.append(data)
    def handle_endtag(self,tag):
        if self.active and tag in ("td","th") and self.cell is not None:
            self.row.append(" ".join("".join(self.cell).split())); self.cell=None
        if self.active and tag == "tr": self.rows.append(self.row)
        if self.active and tag == "figure": self.active=False

def verify_primary_html(blob):
    parser=PrimaryTableParser(); parser.feed(blob.decode("utf-8"))
    parsed=[dict(system=r[0],ocr=r[1],docleak_crit=r[5],valid=r[6])
            for r in parser.rows if len(r)==7 and r[5][:1].isdigit()]
    assert parsed == REPORTED, "Primary Table 3 differs from the frozen minimal input."
    return {"numeric_table_verified":True,"checked_html_sha256":hashlib.sha256(blob).hexdigest(),
            "matches_original_html_hash":hashlib.sha256(blob).hexdigest()==PROVENANCE["source_html_sha256"]}

local_cache=Path("../qa/source_cache/leakagebench_2609_02207v1.html")
VERIFY_REMOTE_SOURCE=False
source_check={"numeric_table_verified_in_this_run":False,
              "reason":"Offline minimal inputs are checksum-verified; source cache is optional."}
if local_cache.is_file():
    source_check=verify_primary_html(local_cache.read_bytes())
elif VERIFY_REMOTE_SOURCE:
    with urllib.request.urlopen(PROVENANCE["source_url"],timeout=30) as response:
        source_check=verify_primary_html(response.read())
print(json.dumps(source_check,indent=2))


{
  "numeric_table_verified": true,
  "checked_html_sha256": "0664b482d98c482ddb81f92264a8a0514917ec2ee4f25da74ea85e5681eb2743",
  "matches_original_html_hash": true
}


## Exact aggregate constraints

For system $i$, let $\widehat d_i$ and $\widehat v_i$ denote the published critical DocLeak and overall validity. The unknown counts are integers: $1\le D\le500$ is the common number of critical-applicable pages, $0\le F_i\le D$ is the number of their matching failures, and $0\le B_i\le500$ is the number of invalid outputs over all pages. The explicit nearest-three-decimal constraints are
\[
\left|\frac{F_i}{D}-\widehat d_i\right|\le0.0005,
\qquad
\left|\frac{500-B_i}{500}-\widehat v_i\right|\le0.0005.
\]
Inclusive endpoints admit both half-unit tie conventions. This is a stated formatting assumption, not a rounding routine documented by the source. Every row must admit integer counts under the **same** $D$. Intersecting the 14 rows' compatible critical denominators gives $D\in\{493,\ldots,500\}$; the code then checks the validity and overlap constraints for each feasible completion. It does not substitute a separate denominator for each model in the primary analysis.

Let $A_i$ be the integer number of outputs that are both invalid and critical-applicable. Page invalidity means an unusable output treated as an empty prediction set, not merely an individually rejected box. Such pages fail the specified matching loss, so
\[
\max\{0,B_i-(500-D)\}\le A_i\le\min\{B_i,F_i\}.
\]
The retained applicable subset must be nonempty: $D>A_i$. Its failure proportion is
\[
d_{{\rm valid},+,i}=\frac{F_i-A_i}{D-A_i}.
\]
In the code, $V=500-B_i$ and $I=A_i$. Overall validity $V/500$ does not identify critical-applicable validity $(D-A_i)/D$.

For a displayed critical rate $t/1000$, the first rounding constraint is evaluated without floating-point error as $2|1000F_i-tD|\le D$; the same integer test is applied to the displayed validity and denominator 500. No independent-sampling assumption or binomial confidence calculation is needed. An endpoint is sharp only relative to these published marginals and the stated constraints; additional information could narrow the interval.

In [3]:
@lru_cache(maxsize=None)
def compatible_counts(displayed_thousandths, denominator, formatting):
    t=displayed_thousandths
    if formatting == "nearest":
        return tuple(k for k in range(denominator+1)
                     if 2*abs(1000*k-t*denominator) <= denominator)
    if formatting == "truncation":
        return tuple(k for k in range(denominator+1)
                     if 1000*k >= t*denominator and
                     (t == 1000 or 1000*k < (t+1)*denominator))
    raise ValueError(formatting)

def thousandths(value):
    exact=Fraction(value)*1000
    assert exact.denominator == 1
    return int(exact)

def possible_denominators(rows, formatting):
    return [d for d in range(1,N+1)
            if all(compatible_counts(thousandths(row["docleak_crit"]),d,formatting)
                   for row in rows)]

common_nearest=possible_denominators(REPORTED,"nearest")
common_truncation=possible_denominators(REPORTED,"truncation")
assert common_nearest == list(range(493,501))
assert common_truncation == [500]
for row in REPORTED:
    for formatting in ["nearest","truncation"]:
        assert len(compatible_counts(thousandths(row["valid"]),N,formatting)) == 1
print("Common critical-applicable count under nearest rounding:", common_nearest)
print("Common critical-applicable count under truncation:", common_truncation)


Common critical-applicable count under nearest rounding: [493, 494, 495, 496, 497, 498, 499, 500]
Common critical-applicable count under truncation: [500]


In [4]:
def feasible_states(row, formatting="nearest", use_all_rows=True, fixed_d=None):
    d_values=([fixed_d] if fixed_d is not None else
              possible_denominators(REPORTED if use_all_rows else [row],formatting))
    for d in d_values:
        for f in compatible_counts(thousandths(row["docleak_crit"]),d,formatting):
            for v in compatible_counts(thousandths(row["valid"]),N,formatting):
                for invalid_applicable in range(max(0,d-v),min(N-v,f)+1):
                    retained=d-invalid_applicable
                    if retained == 0: continue
                    failures=f-invalid_applicable
                    ratio=Fraction(failures,retained)
                    assert ratio <= Fraction(f,d)
                    yield {"D":d,"F":f,"V":v,"I":invalid_applicable,
                           "valid_critical_pages":retained,
                           "valid_critical_failures":failures,"ratio":ratio}

def summarize(row, formatting="nearest", use_all_rows=True):
    states=list(feasible_states(row,formatting,use_all_rows))
    assert states
    low=min(states,key=lambda s:(s["ratio"],s["D"],s["I"]))
    high=max(states,key=lambda s:(s["ratio"],s["D"],s["I"]))
    result={"system":row["system"],"formatting":formatting,
            "denominator_information":"all_14_rows" if use_all_rows else "own_row_only",
            "reported_valid":row["valid"],"reported_critical_docleak":row["docleak_crit"],
            "all_pages":N,"valid_all_pages":low["V"],
            "critical_pages_min":min(s["D"] for s in states),
            "critical_pages_max":max(s["D"] for s in states),
            "valid_critical_pages_min":min(s["valid_critical_pages"] for s in states),
            "valid_critical_pages_max":max(s["valid_critical_pages"] for s in states),
            "lower_bound":float(low["ratio"]),"upper_bound":float(high["ratio"]),
            "lower_bound_fraction":str(low["ratio"]),"upper_bound_fraction":str(high["ratio"]),
            "feasible_count_completions":len(states),
            "interpretation":"finite_table_typed_IoU_matching_not_confidence_or_containment"}
    return result,low,high

def save_csv(name,rows):
    with (OUT/name).open("w",newline="",encoding="utf-8") as handle:
        writer=csv.DictWriter(handle,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)

main_results=[];witnesses=[]
for row in REPORTED:
    summary,low,high=summarize(row)
    main_results.append(summary)
    for endpoint,state in [("lower",low),("upper",high)]:
        witnesses.append({"system":row["system"],"endpoint":endpoint,
                          "status":"compatible_count_completion_not_observed_page_data",
                          **{key:value for key,value in state.items() if key != "ratio"},
                          "exact_ratio":str(state["ratio"])})
save_csv("validity_gate_bounds.csv",main_results)
save_csv("attaining_completions.csv",witnesses)
for r in main_results:
    print(f"{r['system']}: {100*r['lower_bound']:.6f}% to {100*r['upper_bound']:.6f}%; "
          f"valid critical denominator {r['valid_critical_pages_min']}..{r['valid_critical_pages_max']}")


Presidio: 98.981670% to 98.995984%; valid critical denominator 491..498
Presidio-best (GLiNER restr.): 98.580122% to 98.600000%; valid critical denominator 493..500
GLiNER-base: 97.768763% to 97.800000%; valid critical denominator 493..500
GLiNER-multi-PII: 97.160243% to 97.200000%; valid critical denominator 493..500
NVIDIA GLiNER PII: 98.580122% to 98.600000%; valid critical denominator 493..500
GLiNER2: 99.390244% to 99.398798%; valid critical denominator 492..499
Amazon Comprehend PII: 96.754564% to 96.800000%; valid critical denominator 493..500
OpenAI Privacy Filter: 98.737374% to 98.759305%; valid critical denominator 396..403
Google DLP: 100.000000% to 100.000000%; valid critical denominator 493..500
Qwen3-VL-32B: 99.693252% to 99.699700%; valid critical denominator 326..333
InternVL3-38B: 100.000000% to 100.000000%; valid critical denominator 263..270
GPT-5.4: 100.000000% to 100.000000%; valid critical denominator 493..500
GPT-5.5: 98.985801% to 99.000000%; valid critical deno

## Sensitivity and an explicit non-identifiability check

The next output repeats the calculation under nearest rounding versus truncation and under common-denominator information versus a model's own row alone. These are assumption sensitivities, not additional model experiments.

For a worked row, OpenAI Privacy Filter reports $\widehat d=0.990$ and $\widehat v=0.806$. The integer validity constraint identifies $B=97$ invalid outputs, hence $V=403$ valid outputs. Choose the permitted common denominator $D=493$; its failure constraint identifies $F=488$. The overlap constraints give
\[
\max\{0,97-(500-493)\}=90\le A\le97,
\]
with $D>A$ throughout. At $A=97$, the retained failure proportion is $(488-97)/(493-97)=391/396$. At $A=90$, it is $(488-90)/(493-90)=398/403$. These endpoints agree with the exact enumeration. Both completions satisfy the same reported row and the same $D$, yet give different post-gate proportions because the invalid/applicable overlap is not reported. They are mathematical witnesses, not recovered page logs.

A separately established denominator $D=500$ would instead fix $A=B=500-V$ and permit an exact recount. The conditional $D=500$ output demonstrates that additional information can resolve the issue; it does not assert the source's actual denominator. Full-text review did not identify an explicit critical-applicable count, and the critical histogram's first bin combines zero and positive counts. The identification claim is therefore limited to Table 3's published marginals.

In [5]:
sensitivity=[]
for formatting in ["nearest","truncation"]:
    for use_all in [True,False]:
        for row in REPORTED:
            summary,_,_=summarize(row,formatting,use_all)
            sensitivity.append(summary)
save_csv("sensitivity_bounds.csv",sensitivity)

privacy_row=next(row for row in REPORTED if row["system"]=="OpenAI Privacy Filter")
fixed_states=list(feasible_states(privacy_row,fixed_d=493))
fixed_low=min(fixed_states,key=lambda s:s["ratio"])
fixed_high=max(fixed_states,key=lambda s:s["ratio"])
assert fixed_low["F"]==fixed_high["F"]==488
assert fixed_low["V"]==fixed_high["V"]==403
assert fixed_low["I"]==97 and fixed_high["I"]==90
assert fixed_low["ratio"]==Fraction(391,396)
assert fixed_high["ratio"]==Fraction(398,403)
fixed_witnesses=[]
for label,state in [("completion_A",fixed_low),("completion_B",fixed_high)]:
    fixed_witnesses.append({"system":"OpenAI Privacy Filter","completion":label,
        "status":"compatible_completion_not_observed_data",
        **{k:v for k,v in state.items() if k!="ratio"},"exact_post_gate_ratio":str(state["ratio"])})
save_csv("fixed_denominator_nonidentifiability.csv",fixed_witnesses)
for row in sensitivity:
    if row["system"] in ["OpenAI Privacy Filter","Qwen3-VL-32B","InternVL3-38B"]:
        print(row["system"],row["formatting"],row["denominator_information"],
              f"{100*row['lower_bound']:.6f}%..{100*row['upper_bound']:.6f}%")
print("Same-denominator completions:",json.dumps(fixed_witnesses,indent=2))

known_D500_results=[]
for row in REPORTED:
    states=list(feasible_states(row,fixed_d=500))
    ratios={state["ratio"] for state in states}
    assert len(ratios)==1
    assert all(state["I"]==N-state["V"] for state in states)
    state=states[0]
    known_D500_results.append({"system":row["system"],
        "status":"conditional_on_separately_known_D500_not_a_reported_denominator",
        "critical_pages":500,"valid_critical_pages":state["valid_critical_pages"],
        "valid_critical_failures":state["valid_critical_failures"],
        "exact_ratio":str(state["ratio"]),"ratio":float(state["ratio"])})
save_csv("known_D500_conditional_recount.csv",known_D500_results)


OpenAI Privacy Filter nearest all_14_rows 98.737374%..98.759305%
Qwen3-VL-32B nearest all_14_rows 99.693252%..99.699700%
InternVL3-38B nearest all_14_rows 100.000000%..100.000000%
OpenAI Privacy Filter nearest own_row_only 0.000000%..99.047619%
Qwen3-VL-32B nearest own_row_only 99.570815%..99.699700%
InternVL3-38B nearest own_row_only 100.000000%..100.000000%
OpenAI Privacy Filter truncation all_14_rows 98.759305%..98.759305%
Qwen3-VL-32B truncation all_14_rows 99.699700%..99.699700%
InternVL3-38B truncation all_14_rows 100.000000%..100.000000%
OpenAI Privacy Filter truncation own_row_only 66.666667%..99.099099%
Qwen3-VL-32B truncation own_row_only 99.699700%..99.699700%
InternVL3-38B truncation own_row_only 100.000000%..100.000000%
Same-denominator completions: [
  {
    "system": "OpenAI Privacy Filter",
    "completion": "completion_A",
    "status": "compatible_completion_not_observed_data",
    "D": 493,
    "F": 488,
    "V": 403,
    "I": 97,
    "valid_critical_pages": 396,
   

## What the public evidence supports

Under the primary full-table and nearest-rounding assumptions, withdrawing invalid outputs leaves critical matching failure between 98.7374% and 98.7593% for OpenAI Privacy Filter, between 99.6933% and 99.6997% for Qwen3-VL-32B, and exactly 100% for InternVL3-38B. Their overall release coverages under the proposed gate are 80.6%, 66.6%, and 54.0%. The corresponding valid critical-page denominators are 396–403, 326–333, and 263–270. Truncation yields the upper endpoints. Using only Qwen's own row still gives 99.5708%–99.6997%; the Privacy Filter own-row bound is much less informative, demonstrating the dependence of that tighter result on common full-schema applicability.

Thus these reported failures cannot be attributed solely to outputs the validity gate would withhold. This bounded conclusion follows from the published counts and protocol; it is not a recount of unavailable individual predictions. The published all-page validity fraction cannot be inserted directly as the applicable-page retention fraction. From Table 3's published marginals, the exact retained rate remains unidentifiable for several systems without the joint validity/applicability counts. Additional information can resolve this: if all 500 pages are separately established to be critical-applicable, the retained counts and rates are exactly recoverable. That conditional calculation is included below, but D=500 is not assumed as a reported fact.

The evidence does **not** establish final-artifact containment, masking utility, downstream task utility, arbitrary score-gate behavior, or deployment risk. It does not criticize LeakageBench for making a guarantee it did not claim. It identifies precisely which narrower post-gate statement its public aggregates support, and which claims would require additional retained artifacts. An exact gate audit should retain per-unit policy applicability, validation outcome, actual output/mask, matching or containment loss, and source/subject grouping as appropriate. The inability of inspected aggregates to identify a joint count is a mathematical finding; failing to locate a public prediction repository is not treated as proof that no such repository exists.

In [6]:
expected={
    "Presidio":(Fraction(486,491),Fraction(493,498)),
    "GLiNER2":(Fraction(489,492),Fraction(496,499)),
    "OpenAI Privacy Filter":(Fraction(391,396),Fraction(398,403)),
    "Qwen3-VL-32B":(Fraction(325,326),Fraction(332,333)),
    "InternVL3-38B":(Fraction(1),Fraction(1)),
}
for name,(low,high) in expected.items():
    row=next(r for r in main_results if r["system"]==name)
    assert Fraction(row["lower_bound_fraction"])==low
    assert Fraction(row["upper_bound_fraction"])==high
assert len([r for r in main_results if r["valid_all_pages"]<N])==5
results={"status":"executed_analysis_of_actual_published_aggregates_not_new_model_measurements",
         "source":PROVENANCE,"source_check":source_check,
         "common_critical_counts_nearest":common_nearest,
         "common_critical_counts_truncation":common_truncation,
         "main_results":main_results,"sensitivity_results":sensitivity,
         "fixed_denominator_nonidentifiability":fixed_witnesses,
         "known_D500_conditional_recount":known_D500_results,
         "python_version":platform.python_version(),"all_assertions_passed":True}
(OUT/"results.json").write_text(json.dumps(results,indent=2)+"\n")
manifest={path.name:hashlib.sha256(path.read_bytes()).hexdigest()
          for path in sorted(INPUT.iterdir()) if path.is_file()}
(OUT/"input_checksums.json").write_text(json.dumps(manifest,indent=2)+"\n")
print("All exact finite-table checks passed. No model was trained or evaluated.")


All exact finite-table checks passed. No model was trained or evaluated.
